# 01 · Pipeline: LLM thuần, RAG trên đường KG, KG-RAG (GNN-RAG, CGAR, CGAR-G)

Notebook tái lập toàn bộ số liệu dùng trong phần trình bày. Mọi cell **bỏ qua nếu output đã có** trên Drive
(`graph_rag/results_gnn/<bước>/results.json`), nên chạy lại không tốn GPU. Xoá thư mục tương ứng để chạy lại thật.

| Bước | Nội dung | Phần trình bày | GPU | Thời gian (A100) |
|---|---|---|---|---|
| 00 | Thiết lập, test, smoke | – | – | 5 phút |
| 01 | Dữ liệu, trần coverage | – | – | 2 phút |
| 02 | Retrieval không học: GNN top-K, SBERT top-K | RAG | – | 15 phút |
| 03 | Huấn luyện CGAR (Stage 1) | KG-RAG | có | 20 phút |
| 08 | Reader: LLM thuần, prompt chat, prompt grounded | LLM thuần, RAG | có | 1 giờ / reader |
| 08c | CGAR-G: escalation, prompt ứng viên, verify | KG-RAG | có | 1 giờ / reader |
| 08d | Reader LoRA | KG-RAG | có | 1.5 giờ / reader |
| 11 | Checklist claim | Kết luận | – | vài giây |

Đo hallucination (QHR, grounded-correct) ở notebook 02. Bảng và ví dụ cho phần trình bày ở notebook 03.

## 00 · Thiết lập và smoke test
Test CPU và một lần chạy nhỏ trên dữ liệu tổng hợp, để chắc toàn bộ chuỗi code chạy được trước khi dùng GPU.

In [ ]:
# ==== 00a. Mount Drive + đường dẫn ====
from google.colab import drive; drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/ThS/Research/colab'   # SỬA NẾU KHÁC
CODE_DIR   = f'{DRIVE_BASE}/seminar2_kgrag'                # folder code này (upload nguyên folder lên Drive)
GRAPH_RAG  = f'{DRIVE_BASE}/graph_rag'                     # nơi đã có dữ liệu, checkpoint và kết quả các lần chạy trước
CKPT       = f'{GRAPH_RAG}/ckpt'                           # gnn_features_*.pkl (notebook 00), lora_*_cand (notebook 01, cell 08d)
RESULTS    = f'{GRAPH_RAG}/results_gnn'                    # kết quả; run_bench bỏ qua thư mục đã có results.json

import os, sys, json, subprocess, time, shutil, glob
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'; sys.dont_write_bytecode = True
for d in glob.glob(f'{CODE_DIR}/**/__pycache__', recursive=True): shutil.rmtree(d, ignore_errors=True)
os.makedirs(RESULTS, exist_ok=True); os.makedirs(CKPT, exist_ok=True)
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
print('code =', os.getcwd(), '| results =', RESULTS)
assert os.path.exists('kgqa_bench/cli.py'), 'chưa upload folder seminar2_kgrag lên Drive'

In [ ]:
# ==== 00b. Cài deps + GPU ====
!pip install -q datasets sentence-transformers transformers accelerate scikit-learn networkx tqdm matplotlib
import torch
print('torch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# HF token (chỉ cần cho Llama gated). Thêm secret HF_TOKEN trong Colab → Secrets.
try:
    from google.colab import userdata
    if userdata.get('HF_TOKEN'): os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN'); print('HF_TOKEN set')
except Exception: print('(no HF_TOKEN secret — dùng mirror NousResearch cho Llama)')

In [ ]:
# ==== 00c. Test + smoke run (CPU, ~2 phút) ====
!python -m pytest -q -p no:cacheprovider tests/ 2>&1 | tail -3
SMOKE = f'{RESULTS}/s2_smoke'
if not os.path.exists(f'{SMOKE}/results.json'):
    cmd = ('python -m kgqa_bench.cli run --train synthetic:60:0:train --eval synthetic:30:1 '
           '--methods gnnrag_all gnnrag_top5 sbert_top5 cgar cgar_esc --embedder hash --device cpu '
           '--cgar-s1-epochs 2 --cgar-s2-steps 0 --cgar-esc-ncand 0 --cgar-esc-frac 0.5 '
           f'--reader dummy --reader-verify --no-retrieval-baseline --quiet --out {SMOKE}')
    print('exit =', subprocess.call(cmd, shell=True))
print(open(f'{SMOKE}/report.md').read()[:1500])

In [ ]:
# ==== 00d. Helper dùng chung + cấu hình ====
import pandas as pd

DATA_CACHE = f'{RESULTS}/data_cache'

def run_bench(out, train=None, evals=(), methods=(), extra='', force=False):
    """Gọi CLI; bỏ qua nếu results.json đã có đủ method (force=True để chạy lại)."""
    if os.path.exists(f'{out}/results.json') and not force:
        R = load_results(out)
        have = {m for by_m in R['summaries'].values() for m in by_m}
        missing = [m for m in methods if not any(h == m or h.startswith(m + '_k') for h in have)]
        stored_evals = list(R.get('config', {}).get('eval', []))
        spec_changed = bool(stored_evals) and sorted(stored_evals) != sorted(evals)
        if not missing and not spec_changed:
            print(f'[skip] {out} đã có results.json'); return R
        why = f'thiếu {missing}' if missing else 'spec eval đổi'
        print(f'[rerun] {out} {why} -> chạy lại (bản cũ đổi tên *_old)')
        old = f'{out}_old'; i = 1
        while os.path.exists(old): i += 1; old = f'{out}_old{i}'
        os.rename(out, old)
    cmd = f'python -m kgqa_bench.cli run --device {DEVICE} --out {out} '
    if train: cmd += f'--train {train} '
    cmd += '--eval ' + ' '.join(evals) + ' --methods ' + ' '.join(methods) + ' ' + extra
    print('$', cmd); t0 = time.time()
    rc = subprocess.call(cmd, shell=True)
    print(f'exit={rc}  {(time.time()-t0)/60:.1f} min')
    return load_results(out)

def load_results(out):
    with open(f'{out}/results.json') as f: return json.load(f)

COLS = ['n','f1','cov','prec','hit_ctx','cost','n_tokens','ece','aurc','sel_acc@80','llm_hit','llm_f1']
def show(R, cols=COLS, methods=None):
    """Bảng tóm tắt; `methods` lọc hàng (thư mục cũ có thể chứa thêm baseline không dùng trong seminar)."""
    for ds, by_m in R['summaries'].items():
        df = pd.DataFrame(by_m).T
        if methods: df = df.loc[[m for m in methods if m in df.index]]
        df = df[[c for c in cols if c in df.columns]]
        print(f'\n=== {ds} ==='); display(df.round(3))

def show_cmp(R, methods=None):
    rows = []
    for ds, pairs in R['comparisons'].items():
        for k, c in pairs.items():
            if not c or not c.get('n'): continue
            if methods and k.split('_vs_')[0] not in methods: continue
            b = c['bootstrap']
            rows.append(dict(dataset=ds, pair=k, n=c['n'], delta=round(b['delta']*100,2),
                             ci=f"[{b['ci_low']*100:+.2f},{b['ci_high']*100:+.2f}]", p=round(b['p_value'],4),
                             mcnemar_p=round(c.get('mcnemar',{}).get('p_value', float('nan')),4), sig=c['significant_0.05']))
    display(pd.DataFrame(rows))

def gate(cond, msg):
    print(('PASS  ' if cond else 'FAIL  ') + msg)

# Dữ liệu: GNN-RAG (ReaRev) features từ notebook 00
_tr, _te, _cwq = f'{CKPT}/gnn_features_train.pkl', f'{CKPT}/gnn_features_webqsp_test.pkl', f'{CKPT}/gnn_features_cwq_test.pkl'
assert os.path.exists(_tr) and os.path.exists(_te), 'thiếu ckpt/gnn_features_*.pkl -> chạy notebooks/00_gnn_features_colab.ipynb'
WEBQSP_TR = f'gnnrag:{_tr}:webqsp:train'
WEBQSP_TE = f'gnnrag:{_te}:webqsp:test'
CWQ_TE    = f'gnnrag:{_cwq}:cwq:test' if os.path.exists(_cwq) else 'rog:rmanluo/RoG-cwq:test'
CGAR_BUDGET   = 5     # pool ReaRev trung bình 5.6 ứng viên: budget 5 đường như Seminar 1
CGAR_S2_STEPS = 0     # chỉ Stage 1 (supervised); Stage 2 RL không cải thiện (ablation)
CGAR_FLAGS    = ''    # bản CGAR gốc; v3 (learned stop, nhãn proof, conf=hit) bị loại trên dev
READER_PROMPTS = ['chat', 'grounded']
EMB_MODEL = 'all-MiniLM-L6-v2'
print('helpers ready | RESULTS =', RESULTS, '| CGAR_BUDGET =', CGAR_BUDGET); print('WEBQSP_TE =', WEBQSP_TE)

## 01 · Dữ liệu và trần coverage
WebQSP test 1639 câu, CWQ test 3531 câu. `answer_in_subgraph_rate` là recall tối đa của mọi phương pháp chọn đường;
"đáp án trong pool ứng viên ReaRev" là trần thực tế của toàn bộ phần KG-RAG.
**Đã đo:** trần subgraph WebQSP 0.887, đáp án trong pool ReaRev 0.826.

In [ ]:
from kgqa_bench.data import load_spec
DATA_CACHE = f'{RESULTS}/data_cache'
stats = {}
for spec in (WEBQSP_TR, WEBQSP_TE, CWQ_TE):
    ds = load_spec(spec, cache_dir=DATA_CACHE)
    stats[ds.key] = ds.stats(); print(ds.key, stats[ds.key])
display(pd.DataFrame(stats).T.round(3))
ceil = stats['webqsp:test']['answer_in_subgraph_rate']; src = str(stats['webqsp:test']['source'])
if src.startswith('gnnrag_pkl'):
    gate(0.80 <= ceil <= 0.95, f'trần subgraph WebQSP(GNN features) = {ceil:.3f} trong [0.80, 0.95] (11/2026: 0.887)')
    import numpy as np
    te_ = load_spec(WEBQSP_TE, cache_dir=DATA_CACHE)
    ph = float(np.mean([any(a in set(q.top_candidates) for a in q.answers) for q in te_ if q.answers]))
    gate(0.65 <= ph <= 0.85, f'đáp án trong pool ứng viên ReaRev (test) = {ph:.3f} (Seminar 1 ≈ 0.73; heuristic ≈ 0.33)')
else:
    gate(ceil >= 0.9, f'trần coverage WebQSP(subgraph RoG) = {ceil:.3f} ≥ 0.9 (lần 11/2026: 0.956; CWQ 0.807)')
    print('Lưu ý: đây là trần của SUBGRAPH. Pool 20 ứng viên heuristic chỉ chứa đáp án ở ~33% câu (hit_ctx gnnrag_all ở bước 02) → scorer là nút thắt, không phải dữ liệu.')
json.dump(stats, open(f'{RESULTS}/b01_dataset_stats.json','w'), indent=2)

## 02 · Retrieval không học (RAG trên đường KG)
Mọi phương pháp dùng chung pool 20 ứng viên và cùng tập đường. `gnnrag_topK` giữ K đường theo điểm GNN; `sbert_topK` giữ K đường
có cosine(câu hỏi, đường) cao nhất, tức RAG dense cổ điển áp dụng lên đường KG đã chuyển thành văn bản.
**Đã đo (WebQSP):** all F1 41.1, hit_ctx 83.2 ở 8.4 đường; sbert_top5 38.5 / 79.0 ở 3.45 đường.

In [ ]:
B02 = f'{RESULTS}/b02_baselines'
M02 = ['gnnrag_all','gnnrag_top10','gnnrag_top5','gnnrag_top3','sbert_top10','sbert_top5']
R02 = run_bench(B02, evals=[WEBQSP_TE, CWQ_TE], methods=M02, extra='--compare-ref gnnrag_top5 --quiet')
show(R02, methods=M02)
s = R02['summaries']['webqsp:test']
gate(s['gnnrag_all']['cov'] > s['gnnrag_top10']['cov'] > s['gnnrag_top5']['cov'] > s['gnnrag_top3']['cov'], 'Recall giảm đơn điệu theo K: cắt đường là cắt bớt đáp án')
gate(s['sbert_top5']['hit_ctx'] < s['gnnrag_all']['hit_ctx'], 'RAG top-5 theo độ tương đồng mất đáp án so với lấy hết đường')

## 03 · Huấn luyện CGAR (Stage 1)
CGAR chấm điểm từng đường bằng mạng nhỏ (relevance head + confidence head) trên embedding SBERT, chọn tối đa 5 đường và trả về
một confidence cho câu hỏi. Checkpoint chọn trên dev tách từ train, không nhìn test.
**Đã đo (WebQSP, 3 seed):** F1 39.2, hit_ctx 79.6 ở 3.45 đường; AURC 0.118, thấp nhất trong mọi phương pháp.

In [ ]:
B03 = f'{RESULTS}/b03_cgar'
R03 = run_bench(B03, train=WEBQSP_TR, evals=[WEBQSP_TE, CWQ_TE], methods=['cgar'],
                extra=f'--dev-split 0.1 --cgar-max-paths {CGAR_BUDGET} --cgar-s1-epochs 10 --cgar-s2-steps {CGAR_S2_STEPS} --compare-ref cgar --quiet')
show(R03)
log = R03['train_logs']['cgar@seed42']
h1 = pd.DataFrame(log['stage1']['history']); display(h1[['epoch','loss','rel_loss','eval_f1','eval_cost']].round(4))
gate(h1['rel_loss'].iloc[-1] < 0.6, 'rel_loss < 0.6 (relevance head học được)')
if 'R02' in globals():
    gate(R03['summaries']['webqsp:test']['cgar']['f1'] >= R02['summaries']['webqsp:test']['gnnrag_top5']['f1'] - 0.005, 'F1 CGAR >= GNN top-5 cùng cost')
CGAR_CKPT = f'{B03}/ckpt/cgar_s1_seed42.pt'
print('CGAR_CKPT =', CGAR_CKPT)

## 08 · Reader: LLM thuần, prompt chat, prompt grounded
1000 câu WebQSP test (subset seed 0), hai reader Qwen2.5-7B-Instruct và Llama-3.1-8B-Instruct.
`no_retrieval` = LLM thuần, không context. Prompt `chat` = prompt gốc của GNN-RAG. Prompt `grounded` = chỉ được trả lời từ đường, cho phép "unknown".
**Đã đo:** LLM thuần 0.474 (Qwen) và 0.588 (Llama) llm_hit; GNN-RAG-all chat 0.765 và 0.805. Prompt grounded giảm bịa với Qwen
nhưng bỏ cuộc 29% câu, với Llama còn bịa nhiều hơn.

In [ ]:
READERS = ['Qwen/Qwen2.5-7B-Instruct', 'NousResearch/Meta-Llama-3.1-8B-Instruct']
# 'mistralai/Mistral-7B-Instruct-v0.3': run chat cũ có trên Drive (b08_reader_Mistral-7B-Instruct-v0.3), không chạy lại
MAX_ITEMS = 1000
R08 = {}
def reader_dir(rd, prompt):   # chat = thư mục cũ (dùng lại); grounded = b08g_*; v3 → hậu tố _v3 vì CGAR khác checkpoint
    return f"{RESULTS}/{'b08' if prompt == 'chat' else 'b08g'}_reader_{rd.split('/')[-1]}" + ('_v3' if CGAR_FLAGS else '')
for rd in READERS:
  for prompt in READER_PROMPTS:
    out = reader_dir(rd, prompt)
    R08[(rd, prompt)] = run_bench(out, evals=[WEBQSP_TE], methods=['gnnrag_all','gnnrag_top5','sbert_top5','cgar'],
                        extra=f'--max-items {MAX_ITEMS} --subset-seed 0 --cgar-ckpt {CGAR_CKPT} --cgar-max-paths {CGAR_BUDGET} {CGAR_FLAGS} '
                              f'--reader hf:{rd}:{prompt} --no-retrieval-baseline --compare-ref gnnrag_all --quiet')
    # rescore offline (idempotent, no GPU): parser D2 + abstention-aware unsupported_rate. Also fixes dirs that were [skip]ped
    # Legacy value kept as llm_unsupported_rate_legacy.
    import json
    for _m in [m for m in sys.modules if m in ('kgqa_bench.rescore', 'kgqa_bench.hallucination', 'qa_text_eval')]: del sys.modules[_m]
    from kgqa_bench import rescore as RS; import qa_text_eval as _qa
    assert hasattr(_qa, 'split_answer'), 'qa_text_eval.py trên Drive là bản cũ'
    RS.rescore_dir(out, verbose=False); R08[(rd, prompt)] = json.load(open(f'{out}/results.json'))
    print('\n#####', rd, '| prompt =', prompt); show(R08[(rd, prompt)], cols=['cost','n_tokens_prompt','hit_ctx','llm_hit','llm_hit1','llm_f1','llm_abstained','llm_unsupported_rate','ece_llm','aurc_llm']); show_cmp(R08[(rd, prompt)])
    s = R08[(rd, prompt)]['summaries']['webqsp:test']
    d_rag = (s['gnnrag_all']['llm_hit'] - s['no_retrieval']['llm_hit']) * 100
    print(f'RAG(all) − no_retrieval = {d_rag:+.1f} pp llm_hit')
    gate(d_rag > 0, 'retrieval giúp reader hơn LLM thuần (không retrieval)')
    gate(s['no_retrieval']['llm_hit'] < s['gnnrag_all']['llm_hit'] or s['gnnrag_top5']['llm_hit'] <= s['gnnrag_all']['llm_hit'] + 0.02, 'top5 ≤ all (nhiều context hơn không làm tệ hơn)')
    gate(s['cgar']['llm_unsupported_rate'] <= s['gnnrag_all']['llm_unsupported_rate'] + 0.01, f"CGAR không tăng unsupported_rate (trên câu đã trả lời, abstention đã tách) so với GNN-RAG-all ({s['cgar']['llm_unsupported_rate']:.3f} vs {s['gnnrag_all']['llm_unsupported_rate']:.3f}; so với no-RAG là tầm thường)")
    gate(s['cgar']['llm_hit'] >= s['sbert_top5']['llm_hit'], 'CGAR ≥ sbert_top5 trên llm_hit')
    c = R08[(rd, prompt)]['comparisons']['webqsp:test'].get('cgar_vs_gnnrag_all:llm_hit', {})
    gate(bool(c) and (c['bootstrap']['delta'] > 0 or not c['significant_0.05']), 'CGAR ≥ hoặc tương đương GNN-RAG-all trên llm_hit')
# D1: prompt grounded có giảm unsupported mà không mất quá nhiều hit? (so cùng reader, cùng qid)
for rd in READERS:
    if (rd, 'chat') in R08 and (rd, 'grounded') in R08:
        a, g = R08[(rd, 'chat')]['summaries']['webqsp:test'], R08[(rd, 'grounded')]['summaries']['webqsp:test']
        for m in ('gnnrag_all', 'cgar'):
            print(f"{rd.split('/')[-1]:>28s} {m:>10s}: unsupported chat {a[m]['llm_unsupported_rate']:.3f} → grounded {g[m]['llm_unsupported_rate']:.3f}"
                  f" | abstained {a[m].get('llm_abstained', 0):.3f} → {g[m].get('llm_abstained', 0):.3f} | llm_hit {a[m]['llm_hit']:.3f} → {g[m]['llm_hit']:.3f}")
        gate(g['cgar']['llm_unsupported_rate'] < a['cgar']['llm_unsupported_rate'] - 0.02, f"{rd.split('/')[-1]}: prompt grounded giảm unsupported của CGAR ≥ 2 pp")
        gate(g['cgar']['llm_hit'] >= a['cgar']['llm_hit'] - 0.03, f"{rd.split('/')[-1]}: prompt grounded không làm CGAR mất quá 3 pp llm_hit")
        # D1 finding 11/2026 (Qwen): grounded prompt -12 pp llm_hit, unsupported không giảm -> reader bỏ cuộc trên câu có evidence (over-abstention), không phải bớt bịa
        ans_g = 1 - g['cgar'].get('llm_abstained', 0)
        print(f"{rd.split('/')[-1]:>28s}       cgar: grounded answered {ans_g:.3f}; hit|answered {g['cgar']['llm_hit']/max(ans_g,1e-9):.3f} (chat {a['cgar']['llm_hit']:.3f})")

## 08c · CGAR-G: ít đường hơn, đúng hơn, ít bịa hơn GNN-RAG-all
**Vấn đề (lần 3 + P9):** CGAR budget 5 thua all đúng ở 17% câu có > 10 ứng viên (F1 20.7 so với 33.4) và bằng/hơn ở phần còn lại; reader zero-shot bỏ sót 4–7 pp câu có đáp án trong context và thêm entity không có path (QHR 0.11–0.14 ở all).
**Giải pháp, ba phần, mỗi phần có ablation riêng:**
1. `cgar_esc` (retrieval): CGAR(5) rồi **escalate** sang toàn bộ path khi > 10 ứng viên hoặc confidence < τ (τ = phân vị 20% trên dev, không nhìn test). Confidence giữ của CGAR.
2. Prompt `cand`: liệt kê **ứng viên = đích của các path** đã đưa, yêu cầu chọn trong danh sách; không cho "unknown" (grounded prompt đã thất bại vì bỏ cuộc).
3. `--reader-verify`: chỉ giữ entity trả lời trùng ứng viên; nếu không còn gì thì lấy ứng viên đầu (path liên quan nhất). Raw text giữ ở `llm_pred_raw`; P9 vẫn đo được trên cả hai.
**So sánh:** hàng tham chiếu là GNN-RAG như công bố (all + prompt chat). Hàng công bằng là all với cùng prompt/verify. Cả hai đều phải in ra.
**Gate:** (a) retrieval: F1(cgar_esc) ≥ F1(all) − 0.3, hit_ctx ≥ all − 1 pp, cost ≤ 80% all; (b) reader: llm_hit(cgar_esc, cand+verify) > llm_hit(all, chat) với CI không chứa 0; (c) công bằng: cgar_esc không thua all cùng cand+verify quá 0.5 pp có ý nghĩa; (d) QHR (P9) của cgar_esc cand+verify < all chat.

In [ ]:
# ==== 08c. CGAR-G ====
# (a) retrieval-only, full test, hai dataset: cgar_esc so với all / top10 / cgar. tau chọn trên dev (carved từ train), ghi vào train_logs.
B06D = f'{RESULTS}/b06d_cgar_esc'
ESC_FLAGS = '--cgar-esc-ncand 10 --cgar-esc-frac 0.2 --cgar-esc-fallback all'
R06D = run_bench(B06D, train=WEBQSP_TR, evals=[WEBQSP_TE, CWQ_TE], methods=['gnnrag_all','gnnrag_top10','sbert_top5','cgar','cgar_esc'],
                 extra=f'--dev-split 0.1 --cgar-ckpt {CGAR_CKPT} --cgar-max-paths {CGAR_BUDGET} {CGAR_FLAGS} {ESC_FLAGS} --compare-ref gnnrag_all --quiet')
show(R06D, cols=['n','f1','cov','prec','hit_ctx','cost','n_tokens','escalated','ece','aurc','sel_acc@80']); show_cmp(R06D)
ESC_TAU = R06D['train_logs'].get('cgar_esc_tau@seed42', {}).get('tau'); print('tau (dev quantile 0.2):', ESC_TAU)
ESC_FLAGS_R = f'--cgar-esc-ncand 10 --cgar-esc-fallback all' + (f' --cgar-esc-tau {ESC_TAU}' if ESC_TAU is not None else '')   # reader runs: cùng tau, không nạp lại train
for dsk in R06D['summaries']:
    s = R06D['summaries'][dsk]; e, a = s['cgar_esc'], s['gnnrag_all']
    print(f'--- {dsk}: cgar_esc F1 {e["f1"]*100:.1f} @ {e["cost"]:.2f} (escalated {e["escalated"]*100:.0f}%) | all {a["f1"]*100:.1f} @ {a["cost"]:.2f} | cgar {s["cgar"]["f1"]*100:.1f} @ {s["cgar"]["cost"]:.2f}')
    gate(e['f1'] >= a['f1'] - 0.003, f'{dsk}: F1 cgar_esc >= all - 0.3 pp')
    gate(e['hit_ctx'] >= a['hit_ctx'] - 0.01, f'{dsk}: hit_ctx cgar_esc >= all - 1 pp ({e["hit_ctx"]:.3f} vs {a["hit_ctx"]:.3f})')
    gate(e['cost'] <= 0.8 * a['cost'], f'{dsk}: cost cgar_esc <= 80% all ({e["cost"]:.2f} vs {a["cost"]:.2f})')

# (b) reader: prompt cand + verify, cùng 1000 câu/seed 0 như 08. Thư mục b08c_*. (c) verify offline trên run chat cũ -> b08v_* (không GPU).
import shutil
from kgqa_bench import rescore as RS
from kgqa_bench.metrics import compare_records
def recs_of(out, method): return [json.loads(l) for l in open(f'{out}/records_webqsp_test_{method}.jsonl')]
R08C, R08V = {}, {}
for rd in READERS:
    tag = rd.split('/')[-1]
    out_c = f'{RESULTS}/b08c_reader_{tag}'
    R08C[rd] = run_bench(out_c, evals=[WEBQSP_TE], methods=['gnnrag_all','gnnrag_top5','sbert_top5','cgar','cgar_esc'],
                         extra=f'--max-items {MAX_ITEMS} --subset-seed 0 --cgar-ckpt {CGAR_CKPT} --cgar-max-paths {CGAR_BUDGET} {CGAR_FLAGS} {ESC_FLAGS_R} '
                               f'--reader hf:{rd}:cand --reader-verify --no-retrieval-baseline --compare-ref gnnrag_all --quiet')
    out_chat = reader_dir(rd, 'chat'); out_v = f'{RESULTS}/b08v_reader_{tag}'
    if os.path.exists(f'{out_chat}/results.json'):
        if not os.path.exists(f'{out_v}/results.json'): shutil.copytree(out_chat, out_v)
        RS.rescore_dir(out_v, verbose=False, verify=True); R08V[rd] = load_results(out_v)
    # bảng tổng hợp: tham chiếu = GNN-RAG như công bố (all, chat); các hàng còn lại cùng 1000 câu
    rows = {}
    if (rd, 'chat') in R08: rows['all | chat (GNN-RAG công bố)'] = R08[(rd, 'chat')]['summaries']['webqsp:test']['gnnrag_all']
    if rd in R08V:
        for m in ('gnnrag_all', 'cgar'): rows[f'{m} | chat+verify'] = R08V[rd]['summaries']['webqsp:test'][m]
    for m in ('gnnrag_all', 'gnnrag_top5', 'sbert_top5', 'cgar', 'cgar_esc'): rows[f'{m} | cand+verify'] = R08C[rd]['summaries']['webqsp:test'][m]
    print('\n#####', tag); display(pd.DataFrame(rows).T[['cost','n_tokens_prompt','hit_ctx','llm_hit','llm_hit1','llm_f1','llm_fallback','llm_abstained','aurc_llm']].round(3))
    # so cặp xuyên thư mục (cùng qid): cgar_esc cand+verify so với all chat (claim chính) và so với all cand+verify (công bằng)
    esc = recs_of(out_c, 'cgar_esc'); allc = recs_of(out_c, 'gnnrag_all')
    pairs = {'cgar_esc(cand+verify) - all(cand+verify)': (esc, allc)}
    if (rd, 'chat') in R08: pairs['cgar_esc(cand+verify) - all(chat, công bố)'] = (esc, recs_of(out_chat, 'gnnrag_all'))
    if rd in R08V: pairs['cgar(chat+verify) - all(chat)'] = (recs_of(out_v, 'cgar'), recs_of(out_chat, 'gnnrag_all'))
    for name, (A, B) in pairs.items():
        for met in ('llm_hit', 'llm_f1'):
            c = compare_records(A, B, met, n_bootstrap=10000); b = c['bootstrap']
            print(f"{name:48s} [{met}] Δ={b['delta']*100:+.2f} CI=[{b['ci_low']*100:+.2f},{b['ci_high']*100:+.2f}] p={b['p_value']:.4f} n={c['n']}")
    sc = R08C[rd]['summaries']['webqsp:test']
    if (rd, 'chat') in R08:
        c = compare_records(esc, recs_of(out_chat, 'gnnrag_all'), 'llm_hit', n_bootstrap=10000)['bootstrap']
        gate(c['ci_low'] > 0, f'{tag}: llm_hit cgar_esc(cand+verify) > all(chat) với CI không chứa 0 (Δ={c["delta"]*100:+.2f})')
    c = compare_records(esc, allc, 'llm_hit', n_bootstrap=10000)['bootstrap']
    gate(c['delta'] >= -0.005 or c['ci_high'] > 0, f'{tag}: công bằng — cgar_esc không thua all cùng cand+verify (Δ={c["delta"]*100:+.2f} [{c["ci_low"]*100:+.2f},{c["ci_high"]*100:+.2f}])')
    gate(sc['cgar_esc']['cost'] <= 0.8 * sc['gnnrag_all']['cost'], f'{tag}: cost cgar_esc {sc["cgar_esc"]["cost"]:.2f} <= 80% all {sc["gnnrag_all"]["cost"]:.2f}')
    gate(sc['cgar_esc']['llm_fallback'] <= 0.05, f'{tag}: fallback (không entity nào trùng ứng viên) <= 5% ({sc["cgar_esc"]["llm_fallback"]:.3f})')
print('QHR / grounded-correct của các thư mục b08c_*, b08v_* -> chạy P9 (tự nhận prefix b08c = cand+verify, b08v = chat+verify).')

## 08d · Reader LoRA (CGAR-G phần 3, tuỳ chọn, ~30 phút/reader trên A100)
GNN-RAG công bố dùng reader fine-tune trên train với path của retriever; reader của ta zero-shot. Fine-tune LoRA trên (câu hỏi, path, ứng viên) → danh sách entity gold có trong ứng viên, **trộn context của all và cgar** để không thiên vị một retriever. Đánh giá lại ma trận 08c với adapter → `b08f_reader_*`. Nếu adapter làm all tăng nhiều hơn cgar_esc thì claim "ít đường hơn" vẫn phải đọc trên bảng này.

In [ ]:
# ==== 08d. Reader LoRA ====
!pip install -q peft
!pip uninstall -y -q torchao 2>/dev/null   # Colab cài sẵn torchao 0.10 quá cũ cho peft mới; không dùng quantization nên gỡ hẳn
# tự kiểm tra: code trên Drive phải là bản có disable_stale_torchao (commit b0f2af7+), nếu không train sẽ chết vì torchao
import importlib, kgqa_bench.llm_reader as _LR; importlib.reload(_LR)
assert hasattr(_LR, 'disable_stale_torchao'), 'kgqa_bench/llm_reader.py trên Drive chưa phải bản mới: sync lại llm_reader.py và reader_ft.py'
!python -c "import peft, importlib.util as u; print('peft', peft.__version__, '| torchao installed:', u.find_spec('torchao') is not None)"
B12 = f'{RESULTS}/b12_train_ctx'
R12 = run_bench(B12, evals=[WEBQSP_TR], methods=['gnnrag_all','cgar'], extra=f'--cgar-ckpt {CGAR_CKPT} --cgar-max-paths {CGAR_BUDGET} {CGAR_FLAGS} --compare-ref gnnrag_all --n-bootstrap 100 --quiet')
SFT = f'{B12}/sft_cand.jsonl'
if not os.path.exists(SFT):
    !python -m kgqa_bench.reader_ft build --records {B12}/records_webqsp_train_cgar.jsonl {B12}/records_webqsp_train_gnnrag_all.jsonl --eval {WEBQSP_TR} --prompt cand --out {SFT}
R08F = {}
for rd in READERS:
    tag = rd.split('/')[-1]; ADA = f'{CKPT}/lora_{tag}_cand'
    if not os.path.exists(f'{ADA}/adapter_config.json'):
        !python -m kgqa_bench.reader_ft train --sft {SFT} --model {rd} --out {ADA} --prompt cand --epochs 2 --lr 1e-4 --r 16 --batch 4 --grad-accum 4 --max-len 2048
    if not os.path.exists(f'{ADA}/adapter_config.json'):
        print(f'[NO] {tag}: adapter chưa train xong ({ADA}), bỏ qua reader run'); continue
    out_f = f'{RESULTS}/b08f_reader_{tag}'
    try:
        R08F[rd] = run_bench(out_f, evals=[WEBQSP_TE], methods=['gnnrag_all','gnnrag_top5','sbert_top5','cgar','cgar_esc'],
                             extra=f'--max-items {MAX_ITEMS} --subset-seed 0 --cgar-ckpt {CGAR_CKPT} --cgar-max-paths {CGAR_BUDGET} {CGAR_FLAGS} {ESC_FLAGS_R} '
                                   f'--reader hf:{rd}:cand --reader-adapter {ADA} --reader-verify --no-retrieval-baseline --compare-ref gnnrag_all --quiet')
    except FileNotFoundError:
        print(f'[NO] {tag}: reader run với adapter thoát lỗi (xem traceback của subprocess ngay phía trên dòng exit=), bỏ qua'); continue
    print('\n#####', tag, '| LoRA cand+verify'); show(R08F[rd], cols=['cost','n_tokens_prompt','hit_ctx','llm_hit','llm_hit1','llm_f1','llm_fallback','aurc_llm']); show_cmp(R08F[rd])
    esc = recs_of(out_f, 'cgar_esc'); allf = recs_of(out_f, 'gnnrag_all')
    for name, B in [('all(cand+verify, LoRA)', allf)] + ([('all(chat, công bố)', recs_of(reader_dir(rd, 'chat'), 'gnnrag_all'))] if (rd, 'chat') in R08 else []):
        c = compare_records(esc, B, 'llm_hit', n_bootstrap=10000)['bootstrap']
        print(f"cgar_esc(LoRA) - {name:28s} [llm_hit] Δ={c['delta']*100:+.2f} CI=[{c['ci_low']*100:+.2f},{c['ci_high']*100:+.2f}] p={c['p_value']:.4f}")
    sf = R08F[rd]['summaries']['webqsp:test']
    gate(sf['cgar_esc']['llm_hit'] >= sf['gnnrag_all']['llm_hit'] - 0.005, f'{tag}: LoRA — cgar_esc >= all - 0.5 pp llm_hit ở {sf["cgar_esc"]["cost"]/sf["gnnrag_all"]["cost"]*100:.0f}% cost')

## 11 · Checklist claim cho phần trình bày
Đọc kết quả đã lưu của notebook 01 và notebook 02 trên Drive. Mỗi claim [OK] phải trích kèm Δ, CI 95%, p từ bảng tương ứng.

In [ ]:
# Checklist claim <-> gate cho phần trình bày: đọc thẳng kết quả đã lưu trên Drive (notebook 01 + notebook 02),
# không phụ thuộc cell nào đã chạy trong runtime này.
# [OK] đạt | [NO] không đạt -> nói rõ là giới hạn | [--] chưa có kết quả trên Drive
import glob, re
def _j(path):
    try:
        with open(path) as f: return json.load(f)
    except (FileNotFoundError, json.JSONDecodeError): return None
def _dirs(prefix): return sorted(d for d in glob.glob(f'{RESULTS}/{prefix}_reader_*') if not re.search(r'_old\d*$', d))
def _sum(d): R = _j(f'{d}/results.json'); return R['summaries'].get('webqsp:test') if R else None
def _hal(d):
    H_ = _j(f'{d}/hallucination/hallucination.json')
    return list(H_['datasets'].values())[0] if H_ and H_.get('datasets') else None
def _pair(pa, pb):   # cùng reader, hai cấu hình
    out = []
    for d in _dirs(pa):
        e = f"{RESULTS}/{pb}_reader_{d.split('_reader_')[-1]}"
        if _hal(d) and os.path.isdir(e) and _hal(e): out.append((_hal(d)['summaries'], _hal(e)['summaries']))
    return out
def _notworse(c): return bool(c) and (c['bootstrap']['delta'] >= 0 or not c['significant_0.05'])
claims = []
_chat = [d for d in _dirs('b08') if _sum(d) and 'no_retrieval' in _sum(d)]
claims.append((f'Phần 1 -> 2: retrieval giúp reader hơn LLM thuần, mọi reader ({len(_chat)})',
               all(_sum(d)['gnnrag_all']['llm_hit'] > _sum(d)['no_retrieval']['llm_hit'] for d in _chat) if _chat else None))
_gp = _pair('b08', 'b08g')
claims.append(('Phần 2: prompt grounded giảm QHR của CGAR >= 2 pp mà grounded-correct giảm <= 3 pp',
               all(g['cgar']['question_hallucination_rate'] <= c['cgar']['question_hallucination_rate'] - 0.02 and
                   g['cgar']['grounded_correct_rate'] >= c['cgar']['grounded_correct_rate'] - 0.03 for c, g in _gp) if _gp else None))
_vp = _pair('b08', 'b08v')
claims.append(('Phần 2/3: verify hậu kỳ giảm QHR của GNN-RAG-all và không giảm grounded-correct',
               all(v['gnnrag_all']['question_hallucination_rate'] < c['gnnrag_all']['question_hallucination_rate'] and
                   v['gnnrag_all']['grounded_correct_rate'] >= c['gnnrag_all']['grounded_correct_rate'] for c, v in _vp) if _vp else None))
_hal_all = {os.path.basename(d): _hal(d) for p in ('b08', 'b08g', 'b08v', 'b08c', 'b08f') for d in _dirs(p)}
_hal_all = {k: v for k, v in _hal_all.items() if v and 'cgar' in v['comparisons']}
def _hnotworse(v, m='cgar'):
    c = v['comparisons'].get(m, {}).get('hallucinated', {}).get('bootstrap', {})
    return bool(c) and (c['delta'] <= 0 or c['p_value'] >= 0.05)
claims.append((f'Phần 3: CGAR (ít đường hơn) không tăng question-hallucination so với GNN-RAG-all ({len(_hal_all)} cấu hình)',
               all(_hnotworse(v) for v in _hal_all.values()) if _hal_all else None))
_ver = {k: v for k, v in _hal_all.items() if re.match(r'b08[vcf]_', k)}
claims.append((f'Phần 3: AURC-incorrect của CGAR thấp hơn GNN-RAG-all ở mọi cấu hình có verify ({len(_ver)})',
               all(v['summaries']['cgar']['aurc_incorrect'] < v['summaries']['gnnrag_all']['aurc_incorrect'] for v in _ver.values()) if _ver else None))
_esc = _j(f'{RESULTS}/b06d_cgar_esc/results.json')
_se = _esc['summaries']['webqsp:test'] if _esc else None
claims.append(('Phần 3: cgar_esc F1 retrieval không thua GNN-RAG-all có ý nghĩa',
               _notworse(_esc['comparisons']['webqsp:test'].get('cgar_esc_vs_gnnrag_all:f1')) if _esc else None))
claims.append(('Phần 3: cgar_esc cost <= 80% GNN-RAG-all', (_se['cgar_esc']['cost'] <= 0.8 * _se['gnnrag_all']['cost']) if _esc else None))
for _p, _lab in (('b08c', 'reader zero-shot cand+verify'), ('b08f', 'reader LoRA cand+verify')):
    _ds = [d for d in _dirs(_p) if _j(f'{d}/results.json')]
    _cm = [_j(f'{d}/results.json')['comparisons']['webqsp:test'].get('cgar_esc_vs_gnnrag_all:llm_hit') for d in _ds]
    claims.append((f'Phần 3: cgar_esc không thua GNN-RAG-all về llm_hit cùng {_lab} ({len(_ds)} reader)',
                   all(_notworse(c) for c in _cm) if _ds else None))
_lc = [(_sum(c), _sum(f)) for c in _dirs('b08c') for f in [f"{RESULTS}/b08f_reader_{c.split('_reader_')[-1]}"] if _sum(c) and _sum(f)]
claims.append((f'Phần 3: reader LoRA tăng llm_f1 của GNN-RAG-all so với zero-shot cùng prompt ({len(_lc)} reader)',
               all(f['gnnrag_all']['llm_f1'] > c['gnnrag_all']['llm_f1'] for c, f in _lc) if _lc else None))
_tag = {True: '[OK] ', False: '[NO] ', None: '[--] '}
_note = {True: '', False: '  -> nói rõ là giới hạn', None: '  (chưa có kết quả trên Drive)'}
for name, ok in claims:
    ok = None if ok is None else bool(ok)
    print(_tag[ok] + name + _note[ok])